<a href="https://colab.research.google.com/github/MarkusThill/techdays26/blob/main/TDConnect4Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install bitbully[gui]

In [1]:
# Only relevant for Google Colab:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import output

    output.enable_custom_widget_manager()

# Export Legacy Format to Standardized format:

In [2]:
from techdays26.legacy_ntuple_agent import TDWeightsLoader, export_two_player_weights_zip, import_two_player_weights_zip
from techdays26.legacy_ntuple_agent import TDConnect4Agent, TDEvaluator
import numpy as np

In [3]:
# =============================================================================
# Example usage for legacy weights in ZIP file (including Protocol check)
# =============================================================================
if False:
  loader = TDWeightsLoader(int_dtype=np.int32, validate=True, strict_t=True)

  # Directory:
  # both = loader.load_two_player(".")

  # Zip:
  both = loader.load_two_player_from_zip("C4.TCL-EXP-lam0.5-et.txt.zip", p0="p0.txt", p1="p1.txt")

  evaluator = TDEvaluator(both)
  td_agent = TDConnect4Agent(evaluator, tie_break="center")

In [4]:
if True:
  loader = TDWeightsLoader(int_dtype=np.int32, validate=True, strict_t=True)
  both = loader.load_two_player_from_zip("C4.TCL-EXP-lam0.5-et.txt.zip", p0="p0.txt", p1="p1.txt")

  # Export to a clean single file:
  export_two_player_weights_zip("td_weights_clean.tdw.zip", both)

evaluator = TDEvaluator(import_two_player_weights_zip("td_weights_clean.tdw.zip", int_dtype=np.int32))
td_agent = TDConnect4Agent(evaluator, tie_break="center")

In [5]:
# Structural/runtime Protocol check (optional):
from bitbully.agent_interface import Connect4Agent
assert isinstance(td_agent, Connect4Agent)

In [6]:
from bitbully import BitBully
bitbully_agent = BitBully(opening_book="12-ply-dist", tie_break="random")

In [7]:
agents: dict[str, Connect4Agent] = {
    "BitBully-12ply": bitbully_agent,
    "TD-Agent": td_agent,
}

In [ ]:
%matplotlib ipympl
from bitbully.gui_c4 import GuiC4

c4gui = GuiC4(agents=agents, autoplay=True)

# Display everything
display(c4gui.get_widget())

In [ ]:
from techdays26.legacy import play_match

result = play_match(td_agent, bitbully_agent, verbose=1)
# 0 draw, 1 yellow win, 2 red win
print(result)

ImportError: cannot import name 'bitbully' from 'bitbully' (/home/mthill/techdays26/venv/lib/python3.12/site-packages/bitbully/__init__.py)

In [ ]:
from collections import defaultdict

n_matches = 200
total_scores = defaultdict(int)
for _ in range(n_matches):
    result = play_match(td_agent, bitbully_agent, verbose=0)
    total_scores[result] += 1

In [ ]:
total_scores

In [ ]:
# -----------------------------
# 2) Arena config
# -----------------------------

# Logger is optional; arena is silent by default unless you configure logging.
logger = logging.getLogger("bitbully.arena")
logger.setLevel(logging.WARNING)  # warnings for illegal/exception/timeout
# If you want to see logs in console:
# logging.basicConfig(level=logging.WARNING)

cfg = ArenaConfig(
    agents=(
        AgentSpec(
            agent_id="random_agent",
            agent=RandomAgent(),
            colors=(Color.YELLOW, Color.RED),  # can play both
            epsilons=(0.00,),
        ),
        AgentSpec(
            agent_id="bitbully_12plydist_random",
            agent=bitbully_agent,
            colors=(Color.YELLOW, Color.RED),  # can play both
            epsilons=(0.00, 0.05, 0.10, 0.15, 0.2, 0.25),
        ),
        AgentSpec(
            agent_id="td_center",
            agent=td_agent,
            colors=(Color.YELLOW, Color.RED),  # can play both,
            epsilons=(0.00,),
        ),
    ),
    n_games=100,  # n games per pairing per ε per color-swap
    time_control=TimeControl(
        per_move_timeout_s=2.0,  # best-effort (measured after return)
        per_game_budget_s=45.0,    # seconds of total think time
    ),
    seed=12345,
    log_scores=False,   # set True to also call score_all_moves() for logging
    use_tqdm=True,      # optional progress bar
    logger=logger,
)

# -----------------------------
# 3) Run arena
# -----------------------------

arena = BitBullyArena()
result = arena.run(cfg)

In [ ]:
# -----------------------------
# 4) Inspect results
# -----------------------------

print("Skipped pairings:", len(result.skipped))
print("Games played:", len(result.games))
print("Aggregate rows:", len(result.aggregates))

# Print aggregate rows (one row per (yellow_agent, red_agent, eps_y, eps_r))
for row in result.aggregates:
    print(row)

# Look at a single game record (includes move list + per-move metadata)
g0 = result.games[0]
print("First game:", g0.game_cfg)
print("Termination:", g0.termination, "winner:", g0.winner)
print("Moves:", g0.moves)
print("First move meta:", g0.move_meta[0])